In [1]:
# ============ 第二步：然后才导入其他库 ============
import random
import numpy as np
import torch
import pandas as pd
import pyterrier as pt
from pyterrier.measures import *
from collections import defaultdict
import ir_datasets
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from transformers import BertTokenizer, BertModel
from tqdm import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
dataset = pt.get_dataset('irds:msmarco-passage/trec-dl-2019/judged')
df_colbert = pt.io.read_results("../ColBERT-PRF-VirtualAppendix/ColBERT E2E/E2E.2019.res.gz")
evalMeasuresDict = pt.Evaluate(
  df_colbert,
  dataset.get_qrels(), 
  metrics=[ nDCG@1, nDCG@10, nDCG@50, nDCG@100 ]
)
print(evalMeasuresDict)

dataset = pt.get_dataset('irds:msmarco-passage/trec-dl-2019/judged')
df_colbert_prf = pt.io.read_results("../ColBERT-PRF-VirtualAppendix/ColBERT-PRF/MSMARCO Passage Res/ColBERT PRF Ranker_beta=1/prf_rank_beta1.2019.res.gz")
evalMeasuresDict = pt.Evaluate(
  df_colbert_prf,
  dataset.get_qrels(), 
  metrics=[ nDCG@1, nDCG@10, nDCG@50, nDCG@100 ]
)
print(evalMeasuresDict)

{'nDCG@1': 0.7325581395348837, 'nDCG@10': 0.6934074729307355, 'nDCG@50': 0.6141665697446085, 'nDCG@100': 0.6029510284363208}
{'nDCG@1': 0.7868217054263567, 'nDCG@10': 0.7351525358046614, 'nDCG@50': 0.6890168638568059, 'nDCG@100': 0.6902757382204006}


In [4]:
def deduplicate_passages_by_similarity(df, similarity_threshold=0.9, verbose=True):
    """
    在每个查询内去重：如果不同docno的passage文本相似度超过阈值，保留分数最高的
    
    Parameters:
    -----------
    df : pd.DataFrame
        必须包含列: qid, docno, score, passage_text
    similarity_threshold : float
        相似度阈值，默认0.9 (95%)
    verbose : bool
        是否显示详细信息
    
    Returns:
    --------
    pd.DataFrame : 去重后的结果
    dict : 统计信息
    """
    
    if verbose:
        print("="*60)
        print("DEDUPLICATING PASSAGES BY SIMILARITY")
        print("="*60)
        print(f"Input: {len(df)} rows")
        print(f"Similarity threshold: {similarity_threshold}")
    
    # 检查必需的列
    required_cols = ['qid', 'docno', 'score', 'passage_text']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")
    
    # 删除passage_text为空的行
    df_clean = df.dropna(subset=['passage_text']).copy()
    if verbose and len(df_clean) < len(df):
        print(f"Removed {len(df) - len(df_clean)} rows with missing passage_text")
    
    # 按qid分组处理
    deduped_dfs = []
    stats = {
        'total_queries': 0,
        'total_removed': 0,
        'queries_with_duplicates': 0,
        'duplicate_examples': []
    }
    
    grouped = df_clean.groupby('qid')
    
    for qid, group in tqdm(grouped, desc="Processing queries", disable=not verbose):
        stats['total_queries'] += 1
        
        if len(group) <= 1:
            deduped_dfs.append(group)
            continue
        
        # 按score降序排列
        group = group.sort_values('score', ascending=False).reset_index(drop=True)
        
        # 标记要保留的行
        keep_mask = np.ones(len(group), dtype=bool)
        
        # 提取passage texts
        passages = group['passage_text'].tolist()
        
        # 计算相似度矩阵
        try:
            vectorizer = TfidfVectorizer(min_df=1, stop_words='english')
            tfidf_matrix = vectorizer.fit_transform(passages)
            similarity_matrix = cosine_similarity(tfidf_matrix)
        except:
            similarity_matrix = np.zeros((len(passages), len(passages)))
            for i in range(len(passages)):
                for j in range(len(passages)):
                    if passages[i] == passages[j]:
                        similarity_matrix[i, j] = 1.0
        
        # 找出重复的文档
        removed_count = 0
        for i in range(len(group)):
            if not keep_mask[i]:
                continue
                
            for j in range(i + 1, len(group)):
                if not keep_mask[j]:
                    continue
                
                if similarity_matrix[i, j] >= similarity_threshold:
                    keep_mask[j] = False
                    removed_count += 1
                    
                    # 保存前几个例子
                    if len(stats['duplicate_examples']) < 5:
                        stats['duplicate_examples'].append({
                            'qid': qid,
                            'kept_docno': group.iloc[i]['docno'],
                            'kept_score': group.iloc[i]['score'],
                            'removed_docno': group.iloc[j]['docno'],
                            'removed_score': group.iloc[j]['score'],
                            'similarity': similarity_matrix[i, j],
                            'text_preview': passages[i][:100]
                        })
        
        if removed_count > 0:
            stats['queries_with_duplicates'] += 1
            stats['total_removed'] += removed_count
        
        deduped_dfs.append(group[keep_mask])
    
    df_deduped = pd.concat(deduped_dfs, ignore_index=True)
    
    if verbose:
        print("\n" + "="*60)
        print("DEDUPLICATION SUMMARY:")
        print("="*60)
        print(f"Total queries processed: {stats['total_queries']}")
        print(f"Queries with duplicates: {stats['queries_with_duplicates']}")
        print(f"Total rows removed: {stats['total_removed']}")
        print(f"Output: {len(df_deduped)} rows")
        print(f"Reduction rate: {(len(df_clean) - len(df_deduped)) / len(df_clean) * 100:.2f}%")
        
        if stats['duplicate_examples']:
            print("\n" + "="*60)
            print("EXAMPLE DUPLICATES FOUND:")
            print("="*60)
            for i, ex in enumerate(stats['duplicate_examples'], 1):
                print(f"\n{i}. Query {ex['qid']}:")
                print(f"   ✓ Kept:    docno={ex['kept_docno']}, score={ex['kept_score']:.4f}")
                print(f"   ✗ Removed: docno={ex['removed_docno']}, score={ex['removed_score']:.4f}")
                print(f"   Similarity: {ex['similarity']:.4f}")
                print(f"   Text: {ex['text_preview']}...")
    
    return df_deduped, stats


# ============================================================
# 添加passage_text
# ============================================================

print("="*60)
print("STEP 1: ADDING PASSAGE TEXTS")
print("="*60)

irds_dataset = ir_datasets.load("msmarco-passage/trec-dl-2019/judged")

# 转换为字符串
df_colbert_prf['docno'] = df_colbert_prf['docno'].astype(str)
df_colbert['docno'] = df_colbert['docno'].astype(str)

# 获取需要的所有docno
all_docnos = set(df_colbert_prf['docno'].unique()) | set(df_colbert['docno'].unique())
print(f"Total unique documents needed: {len(all_docnos)}")

# 构建文档映射
print("Building document map...")
doc_map = {}
found_docnos = set()

for doc in tqdm(irds_dataset.docs_iter(), total=8841823, desc="Loading documents"):
    doc_id_str = str(doc.doc_id)
    if doc_id_str in all_docnos:
        doc_map[doc_id_str] = doc.text
        found_docnos.add(doc_id_str)
        if len(found_docnos) == len(all_docnos):
            print(f"\n✓ Found all {len(all_docnos)} documents!")
            break

print(f"Found {len(doc_map)} / {len(all_docnos)} documents")

# 添加passage_text
df_colbert_prf['passage_text'] = df_colbert_prf['docno'].map(doc_map)
df_colbert['passage_text'] = df_colbert['docno'].map(doc_map)

print(f"\nColBERT-PRF: {df_colbert_prf['passage_text'].notna().sum()} / {len(df_colbert_prf)} with text")
print(f"ColBERT E2E: {df_colbert['passage_text'].notna().sum()} / {len(df_colbert)} with text")

# ============================================================
# 测试：检查特定的docno
# ============================================================

print("\n" + "="*60)
print("STEP 2: CHECKING SPECIFIC DOCNOS (2960563, 4173712, 8521673)")
print("="*60)

test_docnos = ['2960563', '4173712', '8521673']

print("\nSearching in ColBERT-PRF:")
for docno in test_docnos:
    rows = df_colbert_prf[df_colbert_prf['docno'] == docno]
    if len(rows) > 0:
        print(f"  ✓ {docno}: found in {len(rows)} row(s)")
        for _, row in rows.iterrows():
            print(f"    QID: {row['qid']}, Score: {row['score']:.4f}")
            if pd.notna(row['passage_text']):
                print(f"    Text: {row['passage_text'][:80]}...")
    else:
        print(f"  ✗ {docno}: not found")

print("\nSearching in ColBERT E2E:")
for docno in test_docnos:
    rows = df_colbert[df_colbert['docno'] == docno]
    if len(rows) > 0:
        print(f"  ✓ {docno}: found in {len(rows)} row(s)")
        for _, row in rows.iterrows():
            print(f"    QID: {row['qid']}, Score: {row['score']:.4f}")
            if pd.notna(row['passage_text']):
                print(f"    Text: {row['passage_text'][:80]}...")
    else:
        print(f"  ✗ {docno}: not found")

# 找到包含这些docno的查询
test_qids = set()
for docno in test_docnos:
    qids_prf = df_colbert_prf[df_colbert_prf['docno'] == docno]['qid'].unique()
    qids_e2e = df_colbert[df_colbert['docno'] == docno]['qid'].unique()
    test_qids.update(qids_prf)
    test_qids.update(qids_e2e)

if test_qids:
    print(f"\nThese docnos appear in {len(test_qids)} queries: {list(test_qids)}")
    
    for qid in list(test_qids)[:2]:  # 只看前2个查询
        print(f"\n--- Checking duplicates in Query {qid} ---")
        
        # 在PRF中
        prf_docs = df_colbert_prf[df_colbert_prf['qid'] == qid].sort_values('score', ascending=False)
        if len(prf_docs) > 0:
            print(f"  ColBERT-PRF: {len(prf_docs)} documents")
            valid_docs = prf_docs[prf_docs['passage_text'].notna()]
            if len(valid_docs) > 0:
                dup_texts = valid_docs.groupby('passage_text').filter(lambda x: len(x) > 1)
                if len(dup_texts) > 0:
                    print(f"    ⚠️  Found {len(dup_texts)} documents with duplicate texts!")
                    for text, group in dup_texts.groupby('passage_text'):
                        print(f"      Text: {text[:60]}...")
                        for _, row in group.iterrows():
                            print(f"        DocNo: {row['docno']}, Score: {row['score']:.4f}")
                else:
                    print(f"    ✓ No exact duplicate texts")
        
        # 在E2E中
        e2e_docs = df_colbert[df_colbert['qid'] == qid].sort_values('score', ascending=False)
        if len(e2e_docs) > 0:
            print(f"  ColBERT E2E: {len(e2e_docs)} documents")
            valid_docs = e2e_docs[e2e_docs['passage_text'].notna()]
            if len(valid_docs) > 0:
                dup_texts = valid_docs.groupby('passage_text').filter(lambda x: len(x) > 1)
                if len(dup_texts) > 0:
                    print(f"    ⚠️  Found {len(dup_texts)} documents with duplicate texts!")
                    for text, group in dup_texts.groupby('passage_text'):
                        print(f"      Text: {text[:60]}...")
                        for _, row in group.iterrows():
                            print(f"        DocNo: {row['docno']}, Score: {row['score']:.4f}")
                else:
                    print(f"    ✓ No exact duplicate texts")

# ============================================================
# 去重
# ============================================================

print("\n" + "="*60)
print("STEP 3: DEDUPLICATING ColBERT-PRF")
print("="*60)
df_colbert_prf_deduped, stats_prf = deduplicate_passages_by_similarity(
    df_colbert_prf, 
    similarity_threshold=0.9,
    verbose=True
)

print("\n" + "="*60)
print("STEP 4: DEDUPLICATING ColBERT E2E")
print("="*60)
df_colbert_deduped, stats_e2e = deduplicate_passages_by_similarity(
    df_colbert, 
    similarity_threshold=0.9,
    verbose=True
)

# ============================================================
# 保存
# ============================================================

print("\n" + "="*60)
print("STEP 5: SAVING RESULTS")
print("="*60)

df_colbert_prf_deduped.to_csv('df_colbert_prf_deduped.csv', index=False)
df_colbert_deduped.to_csv('df_colbert_deduped.csv', index=False)

print("Saved:")
print("  - df_colbert_prf_deduped.csv")
print("  - df_colbert_deduped.csv")

print("\n" + "="*60)
print("ALL DONE!")
print("="*60)

STEP 1: ADDING PASSAGE TEXTS
Total unique documents needed: 303490
Building document map...


Loading documents: 100%|█████████▉| 8841806/8841823 [00:22<00:00, 386030.12it/s]



✓ Found all 303490 documents!
Found 303490 / 303490 documents

ColBERT-PRF: 43000 / 43000 with text
ColBERT E2E: 310072 / 310072 with text

STEP 2: CHECKING SPECIFIC DOCNOS (2960563, 4173712, 8521673)

Searching in ColBERT-PRF:
  ✓ 2960563: found in 1 row(s)
    QID: 1106007, Score: 67.6860
    Text: Definition of Visceral. Visceral: Referring to the viscera, the internal organs ...
  ✓ 4173712: found in 1 row(s)
    QID: 1106007, Score: 67.6616
    Text: Medical Definition of Visceral. Visceral: Referring to the viscera, the internal...
  ✓ 8521673: found in 1 row(s)
    QID: 1106007, Score: 67.3033
    Text: Visceral: Referring to the viscera, the internal organs of the body, specificall...

Searching in ColBERT E2E:
  ✓ 2960563: found in 1 row(s)
    QID: 1106007, Score: 27.6784
    Text: Definition of Visceral. Visceral: Referring to the viscera, the internal organs ...
  ✓ 4173712: found in 1 row(s)
    QID: 1106007, Score: 27.5983
    Text: Medical Definition of Visceral. Viscer

Processing queries: 100%|██████████| 43/43 [00:05<00:00,  8.52it/s]



DEDUPLICATION SUMMARY:
Total queries processed: 43
Queries with duplicates: 43
Total rows removed: 5004
Output: 37996 rows
Reduction rate: 11.64%

EXAMPLE DUPLICATES FOUND:

1. Query 1037798:
   ✓ Kept:    docno=6060285, score=43.6287
   ✗ Removed: docno=720665, score=42.7762
   Similarity: 0.9826
   Text: A state of the northwest United States in the Pacific Northwest. It was admitted as the 33rd state i...

2. Query 1037798:
   ✓ Kept:    docno=3353333, score=39.5958
   ✗ Removed: docno=13689, score=39.1611
   Similarity: 0.9813
   Text: Robert Knox. Robert Knox, FRSE FRCSE MWS (4 September 1793 – 20 December 1862) was a Scottish anatom...

3. Query 1037798:
   ✓ Kept:    docno=2803567, score=39.3841
   ✗ Removed: docno=2803574, score=39.2345
   Similarity: 0.9280
   Text: Robert Allen Hale. Robert Allan Hale (April 7, 1941 – May 26, 2008) — known as Bobby Hale, as well a...

4. Query 1037798:
   ✓ Kept:    docno=2360249, score=38.9773
   ✗ Removed: docno=2258761, score=37.1480
   S

Processing queries: 100%|██████████| 43/43 [03:24<00:00,  4.75s/it]



DEDUPLICATION SUMMARY:
Total queries processed: 43
Queries with duplicates: 43
Total rows removed: 28538
Output: 281534 rows
Reduction rate: 9.20%

EXAMPLE DUPLICATES FOUND:

1. Query 1037798:
   ✓ Kept:    docno=3353333, score=21.6591
   ✗ Removed: docno=13689, score=21.1477
   Similarity: 0.9793
   Text: Robert Knox. Robert Knox, FRSE FRCSE MWS (4 September 1793 – 20 December 1862) was a Scottish anatom...

2. Query 1037798:
   ✓ Kept:    docno=6578676, score=21.5837
   ✗ Removed: docno=4055265, score=21.0747
   Similarity: 0.9673
   Text: Bryshere Y. Gray, also known as Yazz, is an American actor and rapper. This marks his first televisi...

3. Query 1037798:
   ✓ Kept:    docno=4188834, score=21.3755
   ✗ Removed: docno=4188831, score=20.6780
   Similarity: 0.9034
   Text: Robert Arthur Knox (21 August 1989 – 24 May 2008) was an English actor who portrayed the character o...

4. Query 1037798:
   ✓ Kept:    docno=2803574, score=21.1926
   ✗ Removed: docno=2803567, score=21.1894
  